In [2]:
from dataclasses import dataclass
from typing import Literal

import torch
import torch.nn.functional as F
from torch import nn
from torch.distributed.tensor import DTensor

/home/easyvps/miniconda3/envs/transformer/lib/python3.14/site-packages/torch/_subclasses/functional_tensor.py:368: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /__w/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [6]:
class TopKRouter(nn.Module):
    """
    Generate the expert score across all experts per token and select the top-k experts per token.
    """
    def __init__(self, dim: int, num_experts: int, top_k: int, 
                 score_func: Literal["softmax", "sigmoid"], route_norm: bool, 
                 route_scale: float):
        super().__init__()
        self.dim = dim
        self.num_experts = num_experts
        self.top_k = top_k
        self.score_func = score_func
        self.route_norm = route_norm
        self.route_scale = route_scale
        self.gate = nn.Linear(self.dim, self.num_experts, bias=False)

    def forward(self, x: torch.Tensor, expert_bias: torch.Tensor | None = None
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        score = self.gate(x) # (B*S, D) -> (B*S, num_experts)
        if self.score_func == "softmax":
            score = F.softmax(score, dim=-1)
        elif self.score_func == "sigmoid":
            score = torch.sigmoid(score)
        else:
            raise NotImplementedError(f"Unknown score function {self.score_func}")
    
        if expert_bias is not None:
            _, topk_experts_indices = torch.topk(score + expert_bias, self.top_k, dim=1, sorted=False)
            topk_scores = score.gather(dim=1, index=topk_experts_indices)   # gather gate value from original scores to ensure gate grad is not impacted
        else:
            topk_scores, topk_experts_indices = torch.topk(score, self.top_k, dim=1, sorted=False)

        if self.route_norm:
            topk_scores = topk_scores / (topk_scores.sum(dim=-1, keepdim=True) + 1e-20)
        topk_scores = topk_scores * self.route_scale

        num_tokens_per_expert = torch.histc(
            topk_experts_indices.view(-1).float(),
            bins=self.num_experts,
            min=0,
            max=self.num_experts,
        )  # (E,)

        return topk_experts_indices, topk_scores, num_tokens_per_expert


In [8]:
dim = 512
num_experts = 8
top_k = 2
score_func = 'sigmoid'
route_norm = True
route_scale = 1.0
batch_size = 4
seq_len = 100

torch.manual_seed(123)
x = torch.randn(batch_size * seq_len, dim)
x


tensor([[ 0.3374, -0.1778, -0.3035,  ..., -0.0315, -1.0640,  0.9417],
        [-1.3152, -0.0677, -0.1350,  ..., -0.4840, -0.2713, -0.0774],
        [ 0.5229,  0.1553,  0.5247,  ..., -0.4098,  0.4978, -0.3721],
        ...,
        [-0.1770, -0.8957, -0.5846,  ..., -0.5249, -0.2889, -0.4977],
        [-1.0469, -0.2743,  1.0324,  ..., -1.1306,  0.1238,  0.6058],
        [ 0.1668,  0.5727,  1.3684,  ..., -0.6330,  0.6473, -1.5024]])

In [9]:
router = TopKRouter(dim=dim, num_experts=num_experts, top_k=top_k, score_func=score_func, route_norm=route_norm, route_scale=route_scale)

In [12]:
output = router(x)
output[0].shape

torch.Size([400, 2])

In [14]:
output[0]

tensor([[0, 2],
        [2, 6],
        [2, 6],
        [0, 1],
        [2, 4],
        [3, 7],
        [6, 1],
        [4, 0],
        [5, 4],
        [6, 2],
        [0, 3],
        [3, 4],
        [7, 0],
        [7, 6],
        [7, 1],
        [7, 1],
        [6, 5],
        [2, 7],
        [2, 4],
        [5, 1],
        [4, 1],
        [5, 0],
        [3, 7],
        [4, 2],
        [0, 5],
        [1, 5],
        [6, 1],
        [4, 0],
        [7, 4],
        [3, 5],
        [7, 5],
        [1, 7],
        [4, 3],
        [5, 0],
        [3, 5],
        [7, 2],
        [4, 0],
        [1, 0],
        [7, 2],
        [6, 3],
        [7, 2],
        [6, 3],
        [1, 4],
        [0, 3],
        [6, 0],
        [3, 2],
        [7, 6],
        [1, 6],
        [0, 7],
        [7, 0],
        [4, 7],
        [1, 5],
        [1, 7],
        [7, 3],
        [6, 5],
        [0, 2],
        [7, 6],
        [2, 5],
        [6, 7],
        [4, 6],
        [3, 4],
        [7, 4],
        

In [15]:
output[2]

tensor([107.,  98.,  97.,  96.,  99., 111.,  91., 101.])